# 49 — Adaptive Infrastructure

**Engineering statement:** Adaptive infrastructure turns resource allocation into deployable closed-loop serving.

Notebook 43 specified the controller: operating regime → controller policy → resource allocation. Notebook 49 turns that controller into infrastructure: ingress queues, routing, replica pools, telemetry, autoscaling, fallback paths, and deployment-scale feedback.

## Start here

```text
resource allocation
→ runtime routing
→ replica placement
→ telemetry feedback
→ autoscaling
→ fallback handling
→ adaptive infrastructure
```

Notebook 43 answered:

> Given the active operating regime, how should resources be allocated?

Notebook 49 asks:

> How does that allocation become deployable serving infrastructure?
```

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "49_adaptive_infrastructure"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 15,
    "legend.fontsize": 11,
})

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, width, height, text, fontsize=12, lw=1.7):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=lw,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + width/2, y + height/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=1.7, mutation_scale=14, connectionstyle="arc3"):
    patch = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=mutation_scale,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle
    )
    ax.add_patch(patch)
    return patch

## 1. Repository transition

Notebook 49 continues the second systems arc. Notebook 43 defined the resource-allocation controller; Notebook 49 makes the controller operational.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive infrastructure continues the second systems arc", pad=24)

first = [
    ("00\nContext", 0.04),
    ("07\nVerification\nResource", 0.19),
    ("13\nConfidence\nScheduling", 0.34),
    ("17\nSemi-AR\nDrafting", 0.49),
    ("23\nThroughput\nObjective", 0.64),
    ("29\nHardware\nConstraints", 0.79),
    ("37\nOperating\nRegimes", 0.91),
]
w, h, y = 0.10, 0.12, 0.66
for label, x0 in first:
    rounded_box(ax, (x0, y), w, h, label, fontsize=10)
for (_, x1), (_, x2) in zip(first[:-1], first[1:]):
    arrow(ax, (x1 + w, y + h/2), (x2, y + h/2), lw=1.4, mutation_scale=10)

ax.text(0.5, 0.56, "first arc: mechanism → operating regime", ha="center", va="center", fontsize=13)
ax.plot([0.06, 0.94], [0.51, 0.51], color="black", lw=1.4)

second = [
    ("43\nResource\nAllocation", 0.31),
    ("49\nAdaptive\nInfrastructure", 0.50),
    ("53\nDistributed\nScheduling", 0.69),
    ("59\nCluster\nOptimization", 0.86),
]
w2, h2, y2 = 0.13, 0.12, 0.28
for label, x0 in second:
    rounded_box(ax, (x0, y2), w2, h2, label, fontsize=10)
for (_, x1), (_, x2) in zip(second[:-1], second[1:]):
    arrow(ax, (x1 + w2, y2 + h2/2), (x2, y2 + h2/2), lw=1.4, mutation_scale=10)

ax.text(0.59, 0.15, "second arc: controller → infrastructure → distributed scheduling", ha="center", va="center", fontsize=13)
savefig("49_repository_transition.png")

## 2. Adaptive serving infrastructure map

The resource-allocation controller becomes useful when it reaches runtime components: queues, routers, replicas, telemetry, autoscaling, and policy caches.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive serving infrastructure map", pad=22)

main = [
    ("Requests", 0.05, 0.62, 0.12, 0.10),
    ("Ingress\nqueue", 0.22, 0.62, 0.13, 0.10),
    ("Runtime\nrouter", 0.41, 0.62, 0.14, 0.10),
    ("Replica\npool", 0.64, 0.62, 0.13, 0.10),
    ("Target\nverify", 0.83, 0.62, 0.12, 0.10),
]
for i, (label, x, y, w, h) in enumerate(main):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
    if i < len(main)-1:
        arrow(ax, (x+w, y+h/2), (main[i+1][1], y+h/2))

control = [
    ("Telemetry\nstate", 0.16, 0.34, 0.14, 0.09),
    ("Operating-regime\nclassifier", 0.38, 0.32, 0.20, 0.11),
    ("Autoscaler", 0.67, 0.34, 0.13, 0.09),
    ("Policy\ncache", 0.43, 0.13, 0.12, 0.08),
]
for label, x, y, w, h in control:
    rounded_box(ax, (x, y), w, h, label, fontsize=11)

arrow(ax, (0.48, 0.62), (0.48, 0.43))
arrow(ax, (0.30, 0.385), (0.38, 0.385))
arrow(ax, (0.58, 0.385), (0.67, 0.385))
arrow(ax, (0.735, 0.43), (0.705, 0.62))
arrow(ax, (0.48, 0.32), (0.49, 0.21))
arrow(ax, (0.55, 0.17), (0.64, 0.62), connectionstyle="arc3,rad=-0.35")

ax.text(0.50, 0.04, "Infrastructure turns controller policy into routing, scaling, and verification placement.", ha="center", fontsize=13)
savefig("49_infrastructure_map.png")

## 3. Runtime routing controller

Routing is where regime information enters request handling. Request metadata, confidence profiles, and live system state determine the execution path.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Runtime routing controller", pad=20)

labels = [
    "request\nmetadata",
    "confidence\nprofile",
    "system\nstate",
    "policy\nlookup",
    "route\nassignment",
    "replica\nexecution",
]
xs = np.linspace(0.05, 0.82, len(labels))
w, h, y = 0.13, 0.18, 0.55
for x, label in zip(xs, labels):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+w, y+h/2), (x2, y+h/2))

rounded_box(ax, (0.27, 0.20), 0.46, 0.11, "Routing conditions on request structure and live infrastructure state.", fontsize=12)
savefig("49_runtime_routing_controller.png")

## 4. Telemetry feedback loop

Adaptive infrastructure requires telemetry feedback. Latency, queue depth, memory pressure, utilization, and acceptance rate update the next policy decision.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Telemetry feedback closes the infrastructure loop", pad=20)

nodes = {
    "Request\nstream": (0.12, 0.70, 0.15, 0.10),
    "State\nestimator": (0.42, 0.70, 0.16, 0.10),
    "Routing /\nscheduling": (0.72, 0.70, 0.16, 0.10),
    "Serving\nsystem": (0.72, 0.34, 0.16, 0.10),
    "Telemetry\ncollector": (0.42, 0.34, 0.16, 0.10),
    "Policy\nupdate": (0.12, 0.34, 0.15, 0.10),
}
for label, (x,y,w,h) in nodes.items():
    rounded_box(ax, (x,y), w,h, label, fontsize=12)

arrow(ax, (0.27,0.75), (0.42,0.75))
arrow(ax, (0.58,0.75), (0.72,0.75))
arrow(ax, (0.80,0.70), (0.80,0.44))
arrow(ax, (0.72,0.39), (0.58,0.39))
arrow(ax, (0.42,0.39), (0.27,0.39))
arrow(ax, (0.20,0.44), (0.20,0.70))
arrow(ax, (0.50,0.44), (0.50,0.70), connectionstyle="arc3,rad=-0.18")

ax.text(0.5, 0.15, "Observed latency, memory pressure, queue depth, and acceptance rate update the next policy.", ha="center", fontsize=13)
savefig("49_telemetry_feedback_loop.png")

## 5. Autoscaling under regime changes

Scaling gains are regime-dependent. Balanced regimes scale smoothly; compute-limited regimes need more replicas; latency-protected regimes saturate early.

In [ ]:
replicas = np.arange(1, 13)
series = {
    "balanced": 1.0 - np.exp(-0.42 * replicas),
    "compute-limited": 0.92 * (1.0 - np.exp(-0.23 * replicas)),
    "memory-limited": 0.72 * (1.0 - np.exp(-0.55 * replicas)),
    "latency-protected": 0.60 * (1.0 - np.exp(-0.80 * replicas)) - 0.015 * np.maximum(replicas - 7, 0) ** 1.5,
}

fig, ax = plt.subplots(figsize=(12, 6))
for label, vals in series.items():
    ax.plot(replicas, vals, marker="o", label=label)
ax.set_title("Autoscaling gains depend on operating regime")
ax.set_xlabel("allocated replicas")
ax.set_ylabel("throughput gain proxy")
ax.grid(alpha=0.3)
ax.legend()
savefig("49_autoscaling_regime_gains.png")

## 6. GPU pool allocation by policy

The infrastructure allocator partitions GPU capacity across drafting, verification, reserve headroom, and telemetry overhead.

In [ ]:
regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
parts = ["draft", "verify", "reserve", "telemetry"]
data = np.array([
    [0.38, 0.34, 0.18, 0.10],
    [0.50, 0.25, 0.15, 0.10],
    [0.30, 0.36, 0.24, 0.10],
    [0.28, 0.22, 0.40, 0.10],
])

fig, ax = plt.subplots(figsize=(13, 6))
left = np.zeros(len(regimes))
for j, part in enumerate(parts):
    ax.barh(regimes, data[:, j], left=left, label=part)
    for i, val in enumerate(data[:, j]):
        if val > 0.08:
            ax.text(left[i] + val/2, i, f"{part}\n{val:.2f}", ha="center", va="center", fontsize=10)
    left += data[:, j]
ax.set_xlim(0, 1)
ax.set_xlabel("fraction of GPU pool")
ax.set_title("GPU pool allocation changes by controller policy")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=4)
ax.grid(axis="x", alpha=0.25)
savefig("49_gpu_pool_allocation.png")

## 7. Queue depth versus latency target

A routing controller trades queue depth against verification length and target latency. Tighter latency targets shorten schedules earlier.

In [ ]:
queue = np.arange(4, 65, 4)
lat_targets = [0.8, 1.0, 1.3, 1.7]

fig, ax = plt.subplots(figsize=(12, 6))
for tau in lat_targets:
    sched = np.maximum(1, np.round(8 / (1 + (queue / (18 * tau))**1.4))).astype(int)
    ax.step(queue, sched, where="mid", marker="o", label=f"latency target={tau:.1f}")
ax.set_title("Queue depth shortens scheduled verification under tighter latency")
ax.set_xlabel("queue depth")
ax.set_ylabel("selected verification length ℓ*")
ax.set_ylim(0.5, 8.5)
ax.grid(alpha=0.3)
ax.legend()
savefig("49_queue_latency_schedule.png")

## 8. Replica routing by confidence and load

High-confidence requests can use faster paths. Low-confidence requests need verification-heavy paths. Load shifts the route boundary.

In [ ]:
confidence = np.linspace(0.2, 0.95, 80)
load = np.linspace(0.1, 1.0, 80)
C, L = np.meshgrid(confidence, load)

score = 0.60*C - 0.45*L + 0.18*np.sin(2.6*C)
policy = np.zeros_like(score, dtype=int)
policy[(score > -0.12) & (score <= 0.10)] = 1
policy[(score > 0.10) & (score <= 0.27)] = 2
policy[score > 0.27] = 3

labels = ["safe verify", "mixed route", "fast draft", "fast path"]
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(policy, origin="lower", extent=[confidence.min(), confidence.max(), load.min(), load.max()], aspect="auto")
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(labels)
cbar.set_label("selected route")
ax.set_title("Replica routing depends on confidence and active load")
ax.set_xlabel("request confidence")
ax.set_ylabel("active load")
ax.text(0.33, 0.78, "safe\nverify", ha="center", fontsize=12)
ax.text(0.55, 0.55, "mixed\nroute", ha="center", fontsize=12)
ax.text(0.77, 0.34, "fast\ndraft", ha="center", fontsize=12)
ax.text(0.88, 0.18, "fast\npath", ha="center", fontsize=12)
savefig("49_replica_routing_surface.png")

## 9. Infrastructure phase map

The active infrastructure phase is selected by load and available reserve capacity.

In [ ]:
reserve = np.linspace(0.05, 0.9, 100)
load = np.linspace(0.05, 1.0, 100)
R, L = np.meshgrid(reserve, load)

phase = np.zeros_like(R, dtype=int)
phase[(L > 0.45) & (R > 0.35)] = 1
phase[(L > 0.65) & (R <= 0.35)] = 2
phase[(L <= 0.45) & (R > 0.55)] = 3

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(phase, origin="lower", extent=[reserve.min(), reserve.max(), load.min(), load.max()], aspect="auto")
cbar = plt.colorbar(im, ticks=[0,1,2,3])
cbar.ax.set_yticklabels(["steady", "scale-out", "shed/shorten", "reserve"])
cbar.set_label("infrastructure phase")
ax.set_title("Infrastructure phase map")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
ax.text(0.25, 0.25, "steady", ha="center", fontsize=13)
ax.text(0.62, 0.72, "scale-out", ha="center", fontsize=13)
ax.text(0.18, 0.82, "shed /\nshorten", ha="center", fontsize=13)
ax.text(0.70, 0.25, "reserve", ha="center", fontsize=13)
savefig("49_infrastructure_phase_map.png")

## 10. Saturation fallback path

The infrastructure controller should expose a fallback path rather than silently degrading. Saturation should become observable and recoverable.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Saturation fallback path", pad=20)

main = [
    ("normal\nrouting", 0.06, 0.58),
    ("load\nspike", 0.24, 0.58),
    ("constraint\nactivation", 0.43, 0.58),
    ("shorten\nschedule", 0.64, 0.58),
    ("served\noutput", 0.82, 0.58),
]
w, h = 0.13, 0.12
for i, (label, x, y) in enumerate(main):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
    if i < len(main)-1:
        arrow(ax, (x+w, y+h/2), (main[i+1][1], y+h/2))

rounded_box(ax, (0.43, 0.25), 0.17, 0.11, "fallback:\nqueue / defer", fontsize=12)
rounded_box(ax, (0.67, 0.25), 0.17, 0.11, "telemetry:\nmark degraded", fontsize=12)
arrow(ax, (0.495, 0.58), (0.515, 0.36))
arrow(ax, (0.60, 0.305), (0.67, 0.305))
arrow(ax, (0.755, 0.36), (0.71, 0.58), connectionstyle="arc3,rad=-0.2")
ax.text(0.5, 0.10, "Fallback makes saturation observable and recoverable.", ha="center", fontsize=13)
savefig("49_saturation_fallback_path.png")

## 11. Final synthesis

Adaptive infrastructure bridges notebook-level resource allocation and deployment-scale serving behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Infrastructure specifies adaptive serving", pad=20)

labels = [
    "operating\nregime",
    "controller\npolicy",
    "routing\nstate",
    "replica\nallocation",
    "telemetry\nfeedback",
    "adaptive\nserving",
]
xs = np.linspace(0.05, 0.82, len(labels))
w, h, y = 0.13, 0.18, 0.55
for x, label in zip(xs, labels):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+w, y+h/2), (x2, y+h/2))

rounded_box(ax, (0.25, 0.20), 0.50, 0.11, "Notebook 49 bridges scheduling policy and deployment-scale serving infrastructure.", fontsize=12)
savefig("49_final_synthesis.png")

## Key equations

Infrastructure state:

\[
s_t=(q_t,u_t,m_t,\lambda_t,\alpha_t)
\]

where \(q_t\) is queue state, \(u_t\) utilization, \(m_t\) memory pressure, \(\lambda_t\) latency, and \(\alpha_t\) acceptance rate.

Routing policy:

\[
r_t = R(\rho_t,s_t,b_t)
\]

Replica allocation:

\[
k_t = K(r_t,s_t)
\]

Telemetry update:

\[
s_{t+1}=h(s_t,r_t,k_t,y_t)
\]

Fallback condition:

\[
\text{fallback}=1
\quad\text{if}\quad
q_t>q_{\max}\;\lor\;\lambda_t>\lambda_{\max}\;\lor\;m_t>m_{\max}.
\]

## Notebook summary

Notebook 49 completes the bridge from resource allocation to deployable infrastructure:

- controller policy becomes runtime routing;
- telemetry closes the loop;
- autoscaling and replica routing depend on operating regime;
- fallback paths make saturation observable;
- infrastructure turns confidence scheduling into adaptive serving.

Next notebook: **53 — Distributed Scheduling**.

## Download notebook artifacts

Run the final cell to package Notebook 49 outputs for download.

The archive includes:

- generated `49_*.png` figures,
- manifest JSON,
- the notebook itself when available.

In [ ]:
manifest = {
    "notebook": "49_adaptive_infrastructure_v2.ipynb",
    "title": "Adaptive Infrastructure",
    "engineering_statement": "Adaptive infrastructure turns resource allocation into deployable closed-loop serving.",
    "figures": sorted([p.name for p in FIGURES.glob("49_*.png")]),
    "previous_notebook": "43_resource_allocation.ipynb",
    "next_notebook": "53_distributed_scheduling.ipynb",
}
manifest_path = RESULTS / "49_adaptive_infrastructure_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = RESULTS / "49_adaptive_infrastructure_v2_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIGURES.glob("49_*.png")):
        zf.write(p, arcname=f"figures/{p.name}")

    for p in sorted(RESULTS.glob("*")):
        if p.is_file() and p.resolve() != zip_path.resolve():
            zf.write(p, arcname=f"results/49_adaptive_infrastructure/{p.name}")

    candidates = [
        Path.cwd() / "49_adaptive_infrastructure_v2.ipynb",
        Path.cwd() / "49_adaptive_infrastructure.ipynb",
        REPO_ROOT / "notebooks" / "49_adaptive_infrastructure_v2.ipynb",
        REPO_ROOT / "notebooks" / "49_adaptive_infrastructure.ipynb",
    ]
    for nb_path in candidates:
        if nb_path.exists():
            zf.write(nb_path, arcname=f"notebooks/{nb_path.name}")
            break

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass